## Cell 1: Cài đặt các thư viện cần thiết


In [ ]:
%pip install -q --no-cache-dir \
  "transformers==4.45.0" \
  "peft>=0.13.0,<0.15.0" \
  "datasets>=2.20" \
  "evaluate>=0.4" \
  "accelerate>=0.30" \
  supabase rouge-score sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.5 MB/s eta 0:00:00


In [ ]:
%pip install -U --no-cache-dir bert-score

## Cell 2: Import libraries và thiết lập môi trường

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel
)
from datasets import Dataset, DatasetDict
import evaluate
from supabase import create_client, Client
from google.colab import userdata
from huggingface_hub import login, HfApi

# ─── Seed ───
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── GPU ───
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─── Hằng số ───
MODEL_NAME        = 'VietAI/vit5-base'
MAX_INPUT_LENGTH  = 1024
MAX_TARGET_LENGTH = 300
TRAIN_RATIO       = 0.8
OUTPUT_DIR        = '/colab/working/vit5-finetuned'

print('\n Import và thiết lập hoàn tất!')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

 Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

 Import và thiết lập hoàn tất!


## Cell 3: Kết nối Supabase và tải dữ liệu


In [ ]:
# Cell 3: Kết nối Supabase và tải dữ liệu từ bảng `papers`
try:
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_KEY')
    print('Đã lấy secrets thành công từ Colab!')
except Exception as e:
    print(f'Lỗi khi lấy secrets: {e}')
    print('   Hãy kiểm tra lại Colab Secrets theo hướng dẫn ở trên!')
    raise

# ─── Khởi tạo Supabase client ───
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Kết nối Supabase thành công!')
# ─── 1. Kiểm tra tổng số records (không filter) ───
count_response = supabase.table('papers').select('id', count='exact').execute()
print(f'Tổng số records trong bảng papers: {count_response.count}\n')
# ─── Query dữ liệu từ bảng papers ───
# Chỉ lấy records có cả content và abstract không null
print('\nĐang tải dữ liệu từ Supabase (bảng papers)...')

all_records = []
PAGE_SIZE = 100   # Supabase giới hạn 1000 records/request, dùng pagination
offset = 0

while True:
    try:
        response = (
            supabase.table('papers')
            .select('title, content, abstract')
            .not_.is_('content', 'null')
            .not_.is_('abstract', 'null')
            .neq('content', '')
            .neq('abstract', '')
            .range(offset, offset + PAGE_SIZE - 1)
            .execute()
        )

        batch = response.data
        if not batch:
            break  # Đã lấy hết dữ liệu

        all_records.extend(batch)
        offset += PAGE_SIZE
        print(f'   Đã tải: {len(all_records)} records...', end='\r')

        # Giới hạn 1000 records như yêu cầu
        if len(all_records) >= 1000:
            all_records = all_records[:1000]
            break

    except Exception as e:
        # Xử lý lỗi RLS (Row Level Security) của Supabase
        print(f'\n Lỗi query (có thể do RLS): {e}')
        print('   Thử dùng service_role key thay vì anon key để bypass RLS')
        raise

print(f'Tải xong! Tổng số records hợp lệ: {len(all_records)}')

# ─── Chuyển sang DataFrame để kiểm tra ───
df = pd.DataFrame(all_records)
print(f'\nThống kê dataset:')
print(f'   Số bài báo: {len(df)}')
print(f'   Độ dài content trung bình: {df["content"].str.len().mean():.0f} ký tự')
print(f'   Độ dài abstract trung bình: {df["abstract"].str.len().mean():.0f} ký tự')
print(f'\nMẫu dữ liệu đầu tiên:')
print(f'   Title: {df["title"].iloc[0][:100] if "title" in df.columns else "N/A"}')
print(f'   Content (200 ký tự đầu): {df["content"].iloc[0][:200]}...')
print(f'   Abstract (200 ký tự đầu): {df["abstract"].iloc[0][:200]}...')

Đã lấy secrets thành công từ Colab!
Kết nối Supabase thành công!
Tổng số records trong bảng papers: 1000


Đang tải dữ liệu từ Supabase (bảng papers)...
Tải xong! Tổng số records hợp lệ: 994

Thống kê dataset:
   Số bài báo: 994
   Độ dài content trung bình: 17410 ký tự
   Độ dài abstract trung bình: 1063 ký tự

Mẫu dữ liệu đầu tiên:
   Title: Ảnh hưởng của chất trợ tương hợp glycidyl methacrylate và khoáng sét montmorillonite đến tính chất v
   Content (200 ký tự đầu): Đặt vấn đề Polybutylen adipat terephthalate (PBAT) là một polyester mềm dẻo, có độ dãn dài cao và khả năng phân hủy sinh học trong môi trường ủ phân composite [1]. Vì vậy, PBAT thường được ứng dụng tr...
   Abstract (200 ký tự đầu): Polymer blend trên cơ sở polybutylene adipate terephthalate (PBAT) kết hợp với polylactic acid (PLA) với tính chất cơ lý được cải thiện đã được chế tạo thành công trên máy đùn trục vít, sử dụng chất t...


## Cell 4: Chuẩn bị Dataset cho ViT5

In [ ]:
# Cell 4: Chuẩn bị Dataset và Tokenization (ViT5)

# ─── Load Tokenizer ───
print(f'Đang load tokenizer từ {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size}')


# Chỉ giữ lại bài có body đủ dài sau khi clean
df_clean = df[
    (df['content'].str.len() >= 300) &
    (df['abstract'].str.len() >= 50)
][['content', 'abstract']].dropna().reset_index(drop=True)


print(f' Dataset sau khi làm sạch: {len(df_clean)} mẫu')
print(f'   (Trước khi clean: {len(df)} mẫu)')

# Kiểm tra nhanh: content không còn chứa đáp án
sample = df_clean['content'].iloc[0]
leaked = 'Tóm tắt:' in sample or 'Abstract:' in sample
print(f' Kiểm tra leakage: {" VẪN CÒN!" if leaked else " Sạch rồi"}')
print(f'   200 ký tự đầu content sau khi clean:\n   {sample[:200]}\n')

# ══════════════════════════════════════════════════════════════════════

# Prompt cho ViT5
MAX_TARGET_LENGTH = 300

SUMMARIZATION_PROMPT = (
    "Dựa vào nội dung bài báo khoa học dưới đây, viết một đoạn tóm tắt bằng tiếng Việt. "
    "Đoạn tóm tắt phải có: câu mở đầu nêu rõ mục tiêu nghiên cứu, "
    "các câu trình bày phương pháp và kết quả chính, và câu kết luận nêu ý nghĩa. "
    "CHỈ sử dụng thông tin có trong bài. "
    "Tuyệt đối KHÔNG bịa thêm số liệu, công dụng, thời gian, địa điểm hoặc phương pháp nào không được đề cập::\n\n"
)

def preprocess_function(examples):
    """Tokenize cặp (content, abstract) cho ViT5."""
    prompted_inputs = [SUMMARIZATION_PROMPT + doc for doc in examples["content"]]
    model_inputs = tokenizer(
        prompted_inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
        return_token_type_ids=False
    )
    labels = tokenizer(
        text_target=examples['abstract'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
        return_token_type_ids=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


hf_dataset = Dataset.from_pandas(df_clean)
split_dataset = hf_dataset.train_test_split(test_size=1 - TRAIN_RATIO, seed=SEED)
print(f'   Train: {len(split_dataset["train"])} mẫu')
print(f'   Test:  {len(split_dataset["test"])} mẫu')

print('\n Đang tokenize dataset...')
tokenized_dataset = split_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=['content', 'abstract'],
    desc='Tokenizing'
)

print(f'\n Tokenization hoàn tất!')
train_lengths = [len(x) for x in tokenized_dataset['train']['input_ids']]
print(f'\n Thống kê độ dài token (train):')
print(f'   Min: {min(train_lengths)} | Max: {max(train_lengths)} | Mean: {np.mean(train_lengths):.0f}')


Đang load tokenizer từ VietAI/vit5-base...


spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Tokenizer loaded! Vocab size: 36096
 Dataset sau khi làm sạch: 992 mẫu
   (Trước khi clean: 994 mẫu)
 Kiểm tra leakage:  Sạch rồi
   200 ký tự đầu content sau khi clean:
   Đặt vấn đề Polybutylen adipat terephthalate (PBAT) là một polyester mềm dẻo, có độ dãn dài cao và khả năng phân hủy sinh học trong môi trường ủ phân composite [1]. Vì vậy, PBAT thường được ứng dụng tr

   Train: 793 mẫu
   Test:  199 mẫu

 Đang tokenize dataset...


Tokenizing:   0%|          | 0/793 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/199 [00:00<?, ? examples/s]


 Tokenization hoàn tất!

 Thống kê độ dài token (train):
   Min: 257 | Max: 1024 | Mean: 1013


## Cell 5: Load Model ViT5 + Cấu hình LoRA (PEFT)

LoRA giúp giảm đáng kể số tham số cần train.

In [ ]:
# Cell 5: Load Model ViT5 và áp dụng LoRA
print(f' Đang load model {MODEL_NAME}...')

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)

# Cấu hình generation cho ViT5
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id


total_params = sum(p.numel() for p in base_model.parameters())
print(f' Model loaded! Tổng tham số: {total_params / 1e6:.1f}M')

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q", "v", "k", "o"],
    lora_dropout=0.1,
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.decoder_start_token_id = tokenizer.pad_token_id
model.print_trainable_parameters()


 Đang load model VietAI/vit5-base...


pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

 Model loaded! Tổng tham số: 226.0M
trainable params: 3,538,944 || all params: 229,489,920 || trainable%: 1.5421


## Cell 6: Cấu hình Training Arguments


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    weight_decay=0.01,

    fp16=True,
    optim='adamw_torch_fused',
    predict_with_generate=True,
    generation_max_length=300,
    generation_num_beams=4,

    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rouge2',
    greater_is_better=True,
    save_total_limit=3,

    logging_steps=20,
    report_to='none',
    remove_unused_columns=False,
)
print(' Cấu hình training:')
print(f'   Epochs:               {training_args.num_train_epochs}')
print(f'   Batch size (train):   {training_args.per_device_train_batch_size}')
print(f'   Gradient accum:       {training_args.gradient_accumulation_steps}')
print(f'   Effective batch:      {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Learning rate:        {training_args.learning_rate}')
print(f'   FP16:                 {training_args.fp16}')

 Cấu hình training:
   Epochs:               10
   Batch size (train):   4
   Gradient accum:       4
   Effective batch:      16
   Learning rate:        0.0002
   FP16:                 True


## Cell 7: Khởi tạo Trainer và Bắt đầu Training

In [ ]:
# Cell 7: Khởi tạo Seq2SeqTrainer và bắt đầu training

# ─── Load ROUGE metric ───
rouge_metric = evaluate.load('rouge')

# ─── Hàm tính metrics trong quá trình evaluate ───
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Thay -100 (padding) bằng pad_token_id để decode an toàn
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Loại bỏ chuỗi rỗng
    decoded_preds = [pred.strip() if pred.strip() else "Tóm tắt bài báo khoa học." for pred in decoded_preds]

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )
    return {
        'rouge1': round(result.get('rouge1', 0) * 100, 4),
        'rouge2': round(result.get('rouge2', 0) * 100, 4),
        'rougeL': round(result.get('rougeL', 0) * 100, 4),
    }
# ─── Tạo DataCollator (dynamic padding) ───
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # Padding labels với -100 (bị bỏ qua khi tính loss)
    pad_to_multiple_of=8      # Tối ưu tensor operations
)

# ─── Khởi tạo Seq2SeqTrainer ───
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]  # Dừng sớm nếu không cải thiện
)

print(f'   Tổng steps ước tính: {len(tokenized_dataset["train"]) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}')

train_result = trainer.train()

print(f'   Train Loss cuối: {train_result.training_loss:.4f}')
print(f'   Tổng thời gian:  {train_result.metrics["train_runtime"] / 60:.1f} phút')

# Lưu kết quả training
trainer.save_model(OUTPUT_DIR)
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)
print(f'\n Model đã được lưu tại: {OUTPUT_DIR}')

   Tổng steps ước tính: 490


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
0,3.122700,2.600240,27.795000,7.819500,19.474800
1,2.569500,2.377813,31.400100,9.134300,21.285500
2,2.418200,2.326293,55.096100,23.015500,33.433800
4,2.368100,2.277607,56.110200,24.233500,34.014900
5,2.326500,2.265195,57.480400,25.395700,34.386900
6,2.269300,2.264960,57.615400,24.944500,34.633100
8,2.290700,2.258510,57.754800,25.427600,34.555500
9,2.230900,2.258147,57.753900,25.614100,34.544800


   Train Loss cuối: 2.4818
   Tổng thời gian:  100.6 phút
***** train metrics *****
  epoch                    =     9.8492
  total_flos               =  9019887GF
  train_loss               =     2.4818
  train_runtime            = 1:40:34.43
  train_samples_per_second =      1.314
  train_steps_per_second   =      0.081

 Model đã được lưu tại: /colab/working/vit5-finetuned


##  Cell 8: Đánh giá Model bằng ROUGE

In [ ]:
# Cell 8: Đánh giá Model trên Test Set


# Chạy evaluation
eval_results = trainer.evaluate(
    eval_dataset=tokenized_dataset['test'],
    metric_key_prefix='eval'
)

print('📈 KẾT QUẢ ĐÁNH GIÁ (ROUGE Scores)')

print(f"   ROUGE-1 : {eval_results.get('eval_rouge1', 0):.2f}%")
print(f"   ROUGE-2 : {eval_results.get('eval_rouge2', 0):.2f}%")
print(f"   ROUGE-L : {eval_results.get('eval_rougeL', 0):.2f}%")
print(f"   Eval Loss: {eval_results.get('eval_loss', 0):.4f}")

# ─── Giải thích kết quả ───
print('\n Giải thích ROUGE Score:')
print('   ROUGE-1: Tỷ lệ unigrams (từ đơn) khớp với reference')
print('   ROUGE-2: Tỷ lệ bigrams (cặp từ) khớp với reference')
print('   ROUGE-L: Dựa trên Longest Common Subsequence')

# Lưu kết quả evaluation
trainer.save_metrics('eval', eval_results)
print(f'\n Kết quả đã lưu tại {OUTPUT_DIR}/eval_results.json')

📈 KẾT QUẢ ĐÁNH GIÁ (ROUGE Scores)
   ROUGE-1 : 57.75%
   ROUGE-2 : 25.61%
   ROUGE-L : 34.54%
   Eval Loss: 2.2581

 Giải thích ROUGE Score:
   ROUGE-1: Tỷ lệ unigrams (từ đơn) khớp với reference
   ROUGE-2: Tỷ lệ bigrams (cặp từ) khớp với reference
   ROUGE-L: Dựa trên Longest Common Subsequence

 Kết quả đã lưu tại /colab/working/vit5-finetuned/eval_results.json


## Cell 9: Test Inference - Demo với ví dụ thực tế

In [ ]:
# Cell 9: Test Inference

def generate_summary(text: str, max_new_tokens: int = 300) -> str:
    prompted_text = SUMMARIZATION_PROMPT + text

    inputs = tokenizer(
        prompted_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
        return_tensors='pt',
        return_token_type_ids=False
    ).to(device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=60,
            num_beams=6,
            early_stopping=True,
            no_repeat_ngram_size=4,
            length_penalty=1.2,
            repetition_penalty=1.3,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary.strip()


# ─── Demo trên 3 ví dụ ngẫu nhiên ───
print(' DEMO INFERENCE - Tóm tắt 3 bài báo ngẫu nhiên từ test set')
print('=' * 70)

test_indices = random.sample(range(len(split_dataset['test'])), min(3, len(split_dataset['test'])))

for i, idx in enumerate(test_indices, 1):
    example = split_dataset['test'][idx]
    content      = example['content']
    ground_truth = example['abstract']
    generated    = generate_summary(content)

    single_rouge = rouge_metric.compute(
        predictions=[generated], references=[ground_truth]
    )

    print(f'\n Ví dụ #{i} (index: {idx})')
    print(f'   Input (200 ký tự đầu): {content[:200]}...')
    print(f'   ─────────────────────────────────────')
    print(f' Model sinh ra:  {generated}')
    print(f' Ground truth:   {ground_truth[:300]}')
    print(f' ROUGE-2:        {single_rouge["rouge2"]*100:.2f}%')


 DEMO INFERENCE - Tóm tắt 3 bài báo ngẫu nhiên từ test set

 Ví dụ #1 (index: 163)
   Input (200 ký tự đầu): Mở đầu C. pictus là cây thuộc họ Costaceae có nguồn gốc từ Nam Mỹ, sau đó phát triển mạnh tại châu Á (Ấn Độ, Trung Quốc, Malaysia…) như loài cây cảnh. C. pictus đã được sử dụng trong nhiều bài thuốc c...
   ─────────────────────────────────────
 Model sinh ra:  Nghiên cứu này nhằm tối ưu hóa điều kiện tách chiết saponin steroid từ lá cây Costus pictus bằng phương pháp đáp ứng bề mặt và khảo sát kháng khuẩn của cao chiết thu được bằng phương pháp RSM và khảo sát một số hoạt tính sinh học của cao chiết trong điều kiện tối ưu hóa. Kết quả nghiên cứu cho thấy, cao chiết giàu saponin từ lá C. pictus có tác dụng ức chế sự phát triển của các chủng vi khuẩn Bacillus subtilis, Vibrio sp., E. coli và Salmonella sp. (C. pictus). Cao chiết thu được có hàm lượng saponin steroid cao gấp 10 lần so với nồng độ dung môi (20oC), tỷ lệ nguyên liệu/dung môi: 0,79 g/cm3 (25oC). Sau khi kết thúc thí 

In [ ]:

# Sinh tóm tắt cho tất cả mẫu test
predictions = []
references = []

for i, example in enumerate(split_dataset['test']):
    content = example['content']
    ground_truth = example['abstract']

    generated = generate_summary(content)   # hàm đã có ở Cell 9

    predictions.append(generated)
    references.append(ground_truth)

    if (i + 1) % 50 == 0:
        print(f"   Đã sinh {i+1}/{len(split_dataset['test'])} tóm tắt...")

# Tính BERTScore
bertscore_metric = evaluate.load("bertscore")

results = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    model_type="microsoft/mdeberta-v3-base",
    lang="vi",
    batch_size=16,
    rescale_with_baseline=False,
    num_layers=12
)

print(" KẾT QUẢ BERTSCORE")
print(f"   Precision : {np.mean(results['precision']):.4f}")
print(f"   Recall    : {np.mean(results['recall']):.4f}")
print(f"   F1        : {np.mean(results['f1']):.4f}")

# Lưu kết quả
import json
with open(f"{OUTPUT_DIR}/bertscore_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "bertscore_precision": float(np.mean(results['precision'])),
        "bertscore_recall": float(np.mean(results['recall'])),
        "bertscore_f1": float(np.mean(results['f1'])),
        "model_type": "microsoft/mdeberta-v3-base"
    }, f, ensure_ascii=False, indent=2)

print(f" Đã lưu kết quả BERTScore tại: {OUTPUT_DIR}/bertscore_results.json")

   Đã sinh 50/199 tóm tắt...
   Đã sinh 100/199 tóm tắt...
   Đã sinh 150/199 tóm tắt...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

 KẾT QUẢ BERTSCORE
   Precision : 0.9020
   Recall    : 0.8926
   F1        : 0.8972
 Đã lưu kết quả BERTScore tại: /colab/working/vit5-finetuned/bertscore_results.json


## Cell 11: Lưu Model Cục bộ và Hướng dẫn Sử dụng

**Cách tải model và sử dụng sau này:**

In [ ]:
# Cell 11: Lưu model và hướng dẫn sử dụng

import shutil


LORA_SAVE_PATH = '/colab/working/vit5-lora-adapter'

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f'LoRA adapter đã lưu tại: {LORA_SAVE_PATH}')

shutil.make_archive(
    '/colab/working/vit5-lora-adapter',
    'zip',
    LORA_SAVE_PATH
)
print(f' Đã nén thành: /colab/working/vit5-lora-adapter.zip')

print(f'    Đã load và clean {len(df_clean)} bài báo từ Supabase')
print(f'    Train: {len(split_dataset["train"])} | Test: {len(split_dataset["test"])} mẫu')
print(f'    Fine-tuned ViT5 với LoRA (q, v, k, o)')
print(f'    Đánh giá ROUGE + BERTScore')
print(f'    Model lưu tại: {LORA_SAVE_PATH}')
print(f'\n Tải model về: /colab/working/vit5-lora-adapter.zip')


LoRA adapter đã lưu tại: /colab/working/vit5-lora-adapter
 Đã nén thành: /colab/working/vit5-lora-adapter.zip
    Đã load và clean 992 bài báo từ Supabase
    Train: 793 | Test: 199 mẫu
    Fine-tuned ViT5 với LoRA (q, v, k, o)
    Đánh giá ROUGE + BERTScore
    Model lưu tại: /colab/working/vit5-lora-adapter

 Tải model về: /colab/working/vit5-lora-adapter.zip


In [ ]:
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

HF_ADAPTER_REPO = "CV12323/vit5-summarize"

# Push adapter (LoRA weights, ~30-80MB)
model.push_to_hub(HF_ADAPTER_REPO, private=True)
tokenizer.push_to_hub(HF_ADAPTER_REPO, private=True)
print(f"✅ Adapter pushed: https://huggingface.co/{HF_ADAPTER_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         |  554kB / 14.2MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../tmpa1uyl78t/spiece.model: 100%|##########|  820kB /  820kB            

✅ Adapter pushed: https://huggingface.co/CV12323/vit5-summarize
